In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [2]:
import logging
import ast
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [3]:
customer = "MissPompadour GmbH"
from_date = '20260601' #pd.to_datetime(to_date) - pd.DateOffset(days=3)
to_date = '20260720'
connection = "googleAdwords"
url = return_api_url()
print(f"url = {url}")
workspace_ids = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_ids if acc["name"] == customer]
if len(workspace_id) != 1:
    print(sorted([item["name"] for item in workspace_ids]))
workspace_id = workspace_id[0]
goals = ["checkout_completed"]
google_pmax_conv_percentage: 0.07

In [4]:
connections = call_api_with_accountId(
    f"{url}api/connections/query",
    workspace_id,
    {"name": connection},
    logging
)
if len(connections) != 1:
    raise ValueError(f"Expected 1 source, got {len(connections)}")
connection_id = connections[0]["id"]
connection_id

In [5]:
temp=send_to_innkeepr_api_paginated(
            f"{url}api/signals/usage/query",
            workspace_id,
            {"fromDate": from_date, "toDate": to_date},
            logging
        )
usage = pd.json_normalize(temp)

In [6]:
usage["date"].min(), usage["date"].max()

In [19]:
usage[usage["targetings"].astype("str").str.contains("Innkeepr – PMAX_Zubeh")][["date","targetings"]].values

In [71]:
def extrac_conv_count_from_usage_log(entry, goals, conv_perc=1):
    if entry in [None, np.nan]:
        return entry
    if isinstance(entry, str):
        entry = ast.literal_eval(entry)
    if isinstance(entry, float) or isinstance(entry, int):
        count_conv = int(entry * conv_perc)
        if count_conv < 1:
            return random.randint(0, 1)
        return count_conv
    conversions = [
        item["count"]
        for item in entry
        if "count" in item.keys() and "name" in item.keys() and item["name"] in goals
    ]
    return int(sum(conversions))

In [72]:
filtered = usage[usage["connectionId"] == connection_id]
filtered["session"] = filtered["conversions"].apply(
        lambda x: extrac_conv_count_from_usage_log(x, goals)
    )

In [73]:
count_conv = filtered.groupby("date")["session"].sum().reset_index()
count_conv["count_30d"] = count_conv["session"].rolling(30).sum()
count_conv

In [75]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)
sns.lineplot(data=count_conv, x="date", y="session", errorbar=None, ax=ax)
plt.xticks(rotation=90)
plt.grid(True)


In [78]:
temp = count_conv[(count_conv["date"] >= "20260510") &(count_conv["date"] <= "20260516")]
temp.sum()